# Bronze para Silver — geração distribuída

Uma linha por UC com solar. A Bronze entra como texto. Esta Silver converte potência, injeção e data, corrige a escala acima de 75 kWp quando o rendimento bruto é incompatível com kWp e marca a conexão anterior à ligação da UC.

A média injetada fica na linha para classificar a UC. Ela não é subtraída do consumo nem da fatura. A Gold é quem monta `possui_gd` a partir do mês de `data_conexao`.

As regras estão em `docs/auditoria_dados/silver/silver_geracao_distribuida.md`.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType

catalog = "v-energy"
bronze_gd = f"`{catalog}`.bronze.tb_geracao_distribuida"
bronze_uc = f"`{catalog}`.bronze.tb_unidades_consumidoras"
silver_gd = f"`{catalog}`.silver.tb_geracao_distribuida"
quarentena_gd = f"`{catalog}`.silver.tb_geracao_distribuida_quarentena"

DATA_MIN = "2024-01-01"
DATA_MAX = "2026-06-30"
ESCALA_ACIMA_DE = 75
RAZAO_ESCALA_MAX = 1
AMOSTRA_DATA_INCONSISTENTE = [
    "UC100259",
    "UC102507",
    "UC106046",
    "UC110560",
    "UC119180",
]

MODALIDADE = {
    "GERACAO NA PROPRIA UC": "PROPRIA_UC",
    "AUTOCONSUMO REMOTO": "AUTOCONSUMO_REMOTO",
    "GERACAO COMPARTILHADA": "COMPARTILHADA",
}

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.silver")
print(bronze_gd)
print(silver_gd)


`v-energy`.bronze.tb_geracao_distribuida
`v-energy`.silver.tb_geracao_distribuida


In [ ]:
mapa_modalidade = F.create_map(
    [item for par in MODALIDADE.items() for item in (F.lit(par[0]), F.lit(par[1]))]
)

# A Bronze de UC traz o cabeçalho repetido numa linha: data_ligacao vale o texto "data_ligacao".
# try_cast devolve nulo se a data não converter, em vez de interromper a carga.
df_uc = (
    spark.table(bronze_uc)
    .filter(F.trim(F.col("id_uc")) != F.lit("id_uc"))
    .select(
    F.trim(F.col("id_uc")).alias("uc_id"),
    F.try_to_date(F.trim(F.col("data_ligacao"))).alias("data_ligacao"),
    F.col("classe").alias("classe"),
    F.col("municipio").alias("municipio"),
    F.col("ingestion_datetime").alias("uc_ingestion_datetime"),
    )
)
janela_uc = Window.partitionBy("uc_id").orderBy(
    F.col("uc_ingestion_datetime").desc_nulls_last()
)
df_uc = (
    df_uc.withColumn("_rn", F.row_number().over(janela_uc))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "uc_ingestion_datetime")
)

df = (
    spark.table(bronze_gd)
    .select(
        F.trim(F.col("id_uc")).alias("id_uc"),
        F.trim(F.col("potencia_kwp")).alias("potencia_kwp_raw"),
        F.expr("try_cast(regexp_replace(trim(potencia_kwp), ',', '.') as decimal(12,2))").alias("_potencia_num"),
        F.trim(F.col("energia_injetada_kwh_media_mensal")).alias("energia_raw"),
        F.expr("try_cast(trim(energia_injetada_kwh_media_mensal) as decimal(12,2))").alias("_energia_num"),
        F.trim(F.col("data_conexao")).alias("data_conexao_raw"),
        F.try_to_date(F.trim(F.col("data_conexao"))).alias("data_conexao"),
        F.trim(F.col("modalidade")).alias("modalidade_origem"),
        F.col("source_file"),
        F.col("ingestion_datetime").alias("bronze_ingestion_datetime"),
    )
    .join(df_uc, F.col("id_uc") == F.col("uc_id"), "left")
)

modalidade = mapa_modalidade[F.col("modalidade_origem")]
motivo = (
    F.when(F.col("id_uc").isNull() | (F.col("id_uc") == ""), F.lit("id_uc_nulo"))
    .when(
        F.col("_potencia_num").isNull() | (F.col("_potencia_num") <= 0),
        F.lit("potencia_invalida"),
    )
    .when(
        F.col("_energia_num").isNull() | (F.col("_energia_num") <= 0),
        F.lit("energia_injetada_invalida"),
    )
    .when(F.col("data_conexao").isNull(), F.lit("data_conexao_invalida"))
    .when(modalidade.isNull(), F.lit("modalidade_desconhecida"))
    .when(F.col("uc_id").isNull(), F.lit("uc_inexistente"))
)
df = df.withColumn("modalidade", modalidade).withColumn("motivo_rejeicao", motivo)

df_quarentena = df.filter(F.col("motivo_rejeicao").isNotNull())
df_ok = df.filter(F.col("motivo_rejeicao").isNull())

janela_id = Window.partitionBy("id_uc").orderBy(
    F.col("data_conexao").desc_nulls_last(),
    F.col("bronze_ingestion_datetime").desc_nulls_last(),
)
df_ok = df_ok.withColumn("_rn", F.row_number().over(janela_id)).withColumn(
    "_qtd_id", F.count("*").over(Window.partitionBy("id_uc"))
)
n_descartadas = df_ok.filter(F.col("_rn") > 1).count()
df_uma = df_ok.filter(F.col("_rn") == 1)

razao_bruta = F.col("_energia_num") / F.col("_potencia_num")
erro_escala = (
    (F.col("_potencia_num") > F.lit(ESCALA_ACIMA_DE))
    & (razao_bruta < F.lit(RAZAO_ESCALA_MAX))
)
potencia_corrigida = (
    F.when(erro_escala, F.col("_potencia_num") / F.lit(1000))
    .otherwise(F.col("_potencia_num"))
    .cast(DecimalType(10, 2))
)
fora_da_janela = (F.col("data_conexao") < F.to_date(F.lit(DATA_MIN))) | (
    F.col("data_conexao") > F.to_date(F.lit(DATA_MAX))
)
anterior_a_ligacao = F.col("data_ligacao").isNotNull() & (
    F.col("data_conexao") < F.col("data_ligacao")
)

df_silver = (
    df_uma.withColumn("potencia_kwp", potencia_corrigida)
    .withColumn("fl_correcao_escala", erro_escala)
    .withColumn(
        "energia_injetada_kwh_media_mensal",
        F.col("_energia_num").cast(DecimalType(10, 2)),
    )
    .withColumn(
        "kwh_por_kwp_mes",
        (F.col("_energia_num") / potencia_corrigida).cast(DecimalType(10, 2)),
    )
    .withColumn("fl_data_inconsistente", fora_da_janela | anterior_a_ligacao)
    .withColumn("fl_deduplicado", F.col("_qtd_id") > 1)
    .withColumn("silver_ingestion_datetime", F.current_timestamp())
    .select(
        "id_uc",
        "potencia_kwp_raw",
        "potencia_kwp",
        "fl_correcao_escala",
        "energia_injetada_kwh_media_mensal",
        "kwh_por_kwp_mes",
        "data_conexao",
        "fl_data_inconsistente",
        "modalidade",
        "modalidade_origem",
        "fl_deduplicado",
        "classe",
        "municipio",
        "source_file",
        "bronze_ingestion_datetime",
        "silver_ingestion_datetime",
    )
)

def validar_silver(df_alvo, df_uc_ref, rotulo):
    falhas = []

    def checar(nome, condicao):
        print(f"[{'OK' if condicao else 'FALHA'}] {rotulo} — {nome}")
        if not condicao:
            falhas.append(nome)

    pot_orig = F.expr(
        "try_cast(regexp_replace(trim(potencia_kwp_raw), ',', '.') as decimal(12,2))"
    )
    razao_orig = F.col("energia_injetada_kwh_media_mensal") / pot_orig
    espera_escala = (pot_orig > F.lit(ESCALA_ACIMA_DE)) & (
        razao_orig < F.lit(RAZAO_ESCALA_MAX)
    )
    pot_esperada = (
        F.when(espera_escala, pot_orig / F.lit(1000))
        .otherwise(pot_orig)
        .cast(DecimalType(10, 2))
    )
    rend_esperado = (
        F.col("energia_injetada_kwh_media_mensal") / pot_esperada
    ).cast(DecimalType(10, 2))

    com_uc = df_alvo.join(
        df_uc_ref.select("uc_id", "data_ligacao"),
        F.col("id_uc") == F.col("uc_id"),
        "left",
    )
    espera_data = (
        (F.col("data_conexao") < F.to_date(F.lit(DATA_MIN)))
        | (F.col("data_conexao") > F.to_date(F.lit(DATA_MAX)))
        | (
            F.col("data_ligacao").isNotNull()
            & (F.col("data_conexao") < F.col("data_ligacao"))
        )
    )

    checar(
        "uma linha por id_uc",
        df_alvo.select("id_uc").distinct().count() == df_alvo.count(),
    )
    checar(
        "potência positiva",
        df_alvo.filter(
            F.col("potencia_kwp").isNull() | (F.col("potencia_kwp") <= 0)
        ).count()
        == 0,
    )
    checar(
        "injeção positiva",
        df_alvo.filter(
            F.col("energia_injetada_kwh_media_mensal").isNull()
            | (F.col("energia_injetada_kwh_media_mensal") <= 0)
        ).count()
        == 0,
    )
    checar(
        "modalidade no de-para",
        df_alvo.filter(~F.col("modalidade").isin(list(MODALIDADE.values()))).count()
        == 0,
    )
    checar(
        "flag de escala segue a regra de potência e razão",
        df_alvo.filter(F.col("fl_correcao_escala") != espera_escala).count() == 0,
    )
    checar(
        "valor de potencia_kwp e kwh_por_kwp_mes bate com a transformação",
        df_alvo.filter(
            (F.col("potencia_kwp") != pot_esperada)
            | (F.col("kwh_por_kwp_mes") != rend_esperado)
        ).count()
        == 0,
    )
    checar(
        "flag de data segue a regra de janela e data_ligacao",
        com_uc.filter(F.col("fl_data_inconsistente") != espera_data).count() == 0,
    )

    if falhas:
        raise RuntimeError(f"[ERRO] validação ({rotulo}): " + ", ".join(falhas))


n_bronze = spark.table(bronze_gd).count()
n_silver = df_silver.count()
n_quarentena = df_quarentena.count()

if n_silver + n_quarentena + n_descartadas != n_bronze:
    raise RuntimeError(
        f"[ERRO] Bronze {n_bronze} não fecha com Silver {n_silver} + "
        f"quarentena {n_quarentena} + duplicatas {n_descartadas}."
    )

validar_silver(df_silver, df_uc, "antes de gravar")

if n_quarentena:
    print(f"[AVISO] quarentena: {n_quarentena} linha(s). A Silver das linhas válidas segue.")
    df_quarentena.groupBy("motivo_rejeicao").count().show(truncate=False)
else:
    print("[OK] quarentena: 0")
if n_descartadas:
    print(
        f"[AVISO] duplicatas descartadas: {n_descartadas}. "
        "fl_deduplicado marca a linha que ficou."
    )


In [0]:
df_quarentena_out = df_quarentena.select(
    "id_uc",
    "potencia_kwp_raw",
    "energia_raw",
    "data_conexao_raw",
    "modalidade_origem",
    "source_file",
    "bronze_ingestion_datetime",
    "motivo_rejeicao",
).withColumn("silver_ingestion_datetime", F.current_timestamp())

(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_gd)
)
(
    df_quarentena_out.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(quarentena_gd)
)
print(f"Silver gravada: {silver_gd} ({n_silver} linhas)")
print(f"Quarentena gravada: {quarentena_gd} ({n_quarentena} linhas)")


Silver gravada: `v-energy`.silver.tb_geracao_distribuida (3510 linhas)
Quarentena gravada: `v-energy`.silver.tb_geracao_distribuida_quarentena (0 linhas)


In [ ]:
gravada = spark.table(silver_gd)
if gravada.count() != n_silver:
    raise RuntimeError(
        f"[ERRO] Silver gravada com {gravada.count()} linhas, esperado {n_silver}."
    )
validar_silver(gravada, df_uc, "tabela gravada")

# Conferência informativa da amostra desta entrega. Se uma dessas UCs tiver
# a data corrigida na origem, a carga segue; aqui só avisa.
marcadas_amostra = {
    r["id_uc"]
    for r in gravada.filter(
        F.col("id_uc").isin(AMOSTRA_DATA_INCONSISTENTE) & F.col("fl_data_inconsistente")
    )
    .select("id_uc")
    .collect()
}
if marcadas_amostra == set(AMOSTRA_DATA_INCONSISTENTE):
    print("[OK] amostra histórica: as 5 UCs conhecidas seguem marcadas.")
else:
    print(
        f"[AVISO] amostra histórica mudou: marcadas={sorted(marcadas_amostra)}. "
        "A trava principal já conferiu a regra de datas."
    )

print("Validação da tabela gravada fechou.")
gravada.groupBy("modalidade").count().orderBy("modalidade").show(truncate=False)
